# Chapter 7: Finetuning To Follow Instructions

In [3]:
from importlib.metadata import version, PackageNotFoundError

# Used to check installed package versions and handle missing packages


pkgs = ["numpy", "matplotlib", "torch", "tensorflow", "tqdm", "tiktoken"]

for p in pkgs:
    try:
        print(f"{p} version: {version(p)}")
    except PackageNotFoundError:
        print(f"{p} is NOT installed")

numpy version: 2.0.2
matplotlib version: 3.10.6
torch version: 2.9.1+cpu
tensorflow version: 2.20.0
tqdm version: 4.67.3
tiktoken version: 0.12.0


### 7.1 Introduction to instruction finetuning

In chapter 5, we saw that pretraining an LLM involves a training procedure where it learns to generate one word at a time
Hence, a pretrained LLM is good at text completion, but it is not good at following instructions
In this chapter, we teach the LLM to follow instructions better

----------------------------------------------------------------------------------------------------------

Instruction:

Summarize the following text in one sentence:

“Artificial Intelligence enables machines to learn from data, identify patterns, and make decisions with minimal human intervention.”

Desired response:

Artificial Intelligence allows machines to learn from data and make decisions automatically.

-------------------------------------------------------------------------------------------------------------------------------------

### 7.2 Preparing a dataset for supervised instruction finetuning

In [4]:
import json # Used to read JSON files and convert them into Python objects (list/dict).
import os # Used to check whether a file already exists on our system.
import requests  # Used to send an HTTP request to download the file from the internet.


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()  # 404 error, server error, Network issue
        #Know more about 404 error: https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/404
        
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

# params
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)

print("Number of entries:", len(data))

Number of entries: 1100


In [5]:
data[1]

{'instruction': 'Edit the following sentence for grammar.',
 'input': 'He go to the park every day.',
 'output': 'He goes to the park every day.'}

In [6]:
print(data[1].keys())
print(data[1]['instruction'])
print(data[1]['input'])
print(data[1]['output'])


dict_keys(['instruction', 'input', 'output'])
Edit the following sentence for grammar.
He go to the park every day.
He goes to the park every day.


In [7]:
data[2]

{'instruction': 'Convert 45 kilometers to meters.',
 'input': '',
 'output': '45 kilometers is 45000 meters.'}

In [8]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [9]:
print(format_input(data[1]))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Edit the following sentence for grammar.

### Input:
He go to the park every day.


In [10]:
model_input = format_input(data[2])
desired_response = f"\n\n### Response:\n{data[1]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert 45 kilometers to meters.

### Response:
He goes to the park every day.


In [11]:
# there is 1100 data points, we will use 85% for training, 10% for testing and 5% for validation

train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [12]:
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### 7.3 Organizing data into training batches

In [17]:
#2.1(Format data using prompt template) and 2.2(Tokenise formatted data) is implemented here and below cell

import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [23]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

print(tokenizer.encode("Hello world"))

[50256]
[15496, 995]


In [ ]:
# For step 2.3 (Pad tokenised data and create batches), we will implement a custom collate function
# that will be used in the DataLoader to pad the tokenized sequences to the same length within a batch.

def custom_collate_draft_1(
    batch, # dataloader will pass a batch of tokenized sequences to this function
    pad_token_id=50256,
    device="cpu"
):
    # Find the longest sequence in the batch
    # and increase the max length by +1, which will add one extra
    # padding token below
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to batch_max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        # Via padded[:-1], we remove the extra padded token
        # that has been added via the +1 setting in batch_max_length
        # (the extra padding token will be relevant in later codes)
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    # Convert list of inputs to tensor and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [28]:
input_1 = [0, 1, 2, 3, 4]
input_2 = [4, 5, 6]
input_3 = [1, 4, 7, 2]
batch = (input_1, input_2, input_3)
padded_batch = custom_collate_draft_1(batch)
print(padded_batch)

tensor([[    0,     1,     2,     3,     4],
        [    4,     5,     6, 50256, 50256],
        [    1,     4,     7,     2, 50256]])


In [40]:
# 2.4 (Create target sequences for language modeling) is implemented in the below cell, 
# where we will modify the custom collate function to create target sequences 
# that are shifted by one token compared to the input sequences.
def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs to tensor and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [46]:
input_1 = [0, 1, 2, 3, 4]
input_2 = [4, 5, 6]
input_3 = [1, 4, 7, 2]
batch = (input_1, input_2, input_3)
padded_batch = custom_collate_draft_2(batch)

inp, tar = padded_batch

print("Inputs:")
print(inp)
print()
print("Targets:")
print(tar)

Inputs:
tensor([[    0,     1,     2,     3,     4],
        [    4,     5,     6, 50256, 50256],
        [    1,     4,     7,     2, 50256]])

Targets:
tensor([[    1,     2,     3,     4, 50256],
        [    5,     6, 50256, 50256, 50256],
        [    4,     7,     2, 50256, 50256]])


In [ ]:
# 2.5 (Handle padding tokens in target sequences) 

def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,    
    device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # New: Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # New: Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [50]:
input_1 = [0, 1, 2, 3, 4]
input_2 = [4, 5, 6]
input_3 = [1, 4, 7, 2]
batch = (input_1, input_2, input_3)
 
inp, tar = custom_collate_fn(batch)

print("Inputs:")
print(inp)
print()
print("Targets:")
print(tar)


Inputs:
tensor([[    0,     1,     2,     3,     4],
        [    4,     5,     6, 50256, 50256],
        [    1,     4,     7,     2, 50256]])

Targets:
tensor([[    1,     2,     3,     4, 50256],
        [    5,     6, 50256,  -100,  -100],
        [    4,     7,     2, 50256,  -100]])


### 7.4 Creating data loaders for an instruction dataset

In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
from functools import partial # Used to create a new function with some default arguments from an existing function.

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [55]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)